# Model Reconciliation: Published vs ModelSEED

This notebook compares the published ADP1 metabolic model against the ModelSEED-reconstructed model,
then merges unique published reactions into the ModelSEED model after ATP safety filtering.

**Workflow:**
1. Load both models (published from local JSON, ModelSEED from KBase)
2. Run pFBA and FVA (50% optimal) on both models in pyruvate minimal media
3. Build a comprehensive comparison dataframe of all reactions across both models
4. Identify published-only reactions with genes absent from the ModelSEED model
5. Save the ModelSEED model locally as JSON
6. Merge published reactions into the ModelSEED model; run ATP safe expansion to filter ATP-breaking reactions
7. Run pFBA and FVA on the merged model in pyruvate media
8. Save the final merged model
9. Generate an Escher map with pyruvate fluxes and reaction class badges (source x FVA class)

## Cell 1: Load Both Models

- **Published model**: loaded from `models/FullyTranslatedPublishedModel.json` (already translated to ModelSEED namespace)
- **ModelSEED model**: loaded from KBase workspace via `_legacy.get_model()`
- Both models are saved to datacache for cell independence

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Load published model from local JSON
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
print(f"Published model: {len(pub_mdlutl.model.reactions)} reactions, "
      f"{len(pub_mdlutl.model.metabolites)} metabolites, "
      f"{len(pub_mdlutl.model.genes)} genes")
print(f"  Biomass reaction: bio1")

# Load ModelSEED model from KBase
ms_mdlutl = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
# Remove mRNA_ genes
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)
print(f"ModelSEED model: {len(ms_mdlutl.model.reactions)} reactions, "
      f"{len(ms_mdlutl.model.metabolites)} metabolites, "
      f"{len(ms_mdlutl.model.genes)} genes")
print(f"  Biomass reaction: bio1")

# Save model summaries to datacache
model_info = {
    "published": {
        "reactions": len(pub_mdlutl.model.reactions),
        "metabolites": len(pub_mdlutl.model.metabolites),
        "genes": len(pub_mdlutl.model.genes),
        "biomass_rxn": "bio1"
    },
    "modelseed": {
        "reactions": len(ms_mdlutl.model.reactions),
        "metabolites": len(ms_mdlutl.model.metabolites),
        "genes": len(ms_mdlutl.model.genes),
        "biomass_rxn": "bio1"
    }
}
session.cache.save("ModelReconciliation/model_info", model_info)
print("\nModel info saved to datacache.")

## Cell 2: Run pFBA and FVA on Both Models in Pyruvate Minimal Media

- Set pyruvate minimal media on both models
- Run pFBA to get optimal flux distribution
- Run FVA at 50% optimal growth to classify reaction flux flexibility
- Results cached separately for each model in `ModelReconciliation/` subdirectory

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Load published model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")

# Load ModelSEED model
ms_mdlutl = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

# --- Published model: pFBA + FVA ---
print("=== Published Model ===")
pub_pfba = _legacy.run_fba(pub_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                           objective="MAX{bio1}", run_pfba=True)
print(f"pFBA growth rate: {pub_pfba.objective_value:.6f}")

pub_fva = _legacy.run_fva(pub_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                           objective="MAX{bio1}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(pub_fva)} reactions at 50% optimum")

# Package published results
pub_results = {
    "growth_rate": pub_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in pub_pfba.fluxes.items()},
    "fva": pub_fva
}
session.cache.save("ModelReconciliation/published_pyr_results", pub_results)
print("Published results saved.\n")

# --- ModelSEED model: pFBA + FVA ---
print("=== ModelSEED Model ===")
ms_pfba = _legacy.run_fba(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                           objective="MAX{bio1}", run_pfba=True)
print(f"pFBA growth rate: {ms_pfba.objective_value:.6f}")

ms_fva = _legacy.run_fva(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                          objective="MAX{bio1}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(ms_fva)} reactions at 50% optimum")

# Package modelseed results
ms_results = {
    "growth_rate": ms_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in ms_pfba.fluxes.items()},
    "fva": ms_fva
}
session.cache.save("ModelReconciliation/modelseed_pyr_results", ms_results)
print("ModelSEED results saved.")

## Cell 3: Build Comprehensive Comparison Dataframe

Builds a dataframe listing all reactions across both models with columns:
- **ID**: Reaction ID (standardized for exchanges as `EX_cpd<ID>`)
- **published_pyr_flux**: Flux value and FVA class from published model
- **modelseed_pyr_flux**: Flux value and FVA class from ModelSEED model
- **membership**: `both`, `published`, or `modelseed`
- **extra_published_genes**: Genes in published model not in ModelSEED (with MS rxn if mapped)
- **extra_modelseed_genes**: Genes in ModelSEED model not in published (with pub rxn if mapped)
- **directionality**: Bound directionality comparison (published/MS)
- **equation**: Reaction equation with metabolite names
- **stoichiometry_differences**: Differences in stoichiometry/compartments between models

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Load cached results
pub_results = session.cache.load("ModelReconciliation/published_pyr_results")
ms_results = session.cache.load("ModelReconciliation/modelseed_pyr_results")

# Reload models for structure inspection
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
ms_mdlutl = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

pub_model = pub_mdlutl.model
ms_model = ms_mdlutl.model

# Build exchange ID maps: standardized_id -> original_rxn_id
pub_ex_map = _legacy.get_exchange_map(pub_model)
ms_ex_map = _legacy.get_exchange_map(ms_model)

# Build reverse maps: original_rxn_id -> standardized_id
pub_ex_rev = {v: k for k, v in pub_ex_map.items()}
ms_ex_rev = {v: k for k, v in ms_ex_map.items()}

# Build gene-to-reaction maps for cross-referencing extra genes
pub_gene_rxn_map = build_gene_reaction_map(pub_model)
ms_gene_rxn_map = build_gene_reaction_map(ms_model)

# Collect all reaction IDs using standardized exchange names
def canonical_id(rxn, ex_rev_map):
    if rxn.id in ex_rev_map:
        return ex_rev_map[rxn.id]
    return rxn.id

# Map canonical_id -> original reaction object for each model
pub_rxn_by_cid = {}
for rxn in pub_model.reactions:
    cid = canonical_id(rxn, pub_ex_rev)
    pub_rxn_by_cid[cid] = rxn

ms_rxn_by_cid = {}
for rxn in ms_model.reactions:
    cid = canonical_id(rxn, ms_ex_rev)
    ms_rxn_by_cid[cid] = rxn

all_cids = sorted(set(pub_rxn_by_cid.keys()) | set(ms_rxn_by_cid.keys()))
print(f"Total unique reaction IDs (canonical): {len(all_cids)}")
print(f"  Published only: {len(set(pub_rxn_by_cid.keys()) - set(ms_rxn_by_cid.keys()))}")
print(f"  ModelSEED only: {len(set(ms_rxn_by_cid.keys()) - set(pub_rxn_by_cid.keys()))}")
print(f"  In both: {len(set(pub_rxn_by_cid.keys()) & set(ms_rxn_by_cid.keys()))}")

# Build the comparison rows
rows = []
pub_fluxes = pub_results["fluxes"]
ms_fluxes = ms_results["fluxes"]
pub_fva = pub_results["fva"]
ms_fva = ms_results["fva"]

pub_gene_set = set(str(g) for g in pub_model.genes if not str(g).startswith("mRNA_"))
ms_gene_set = set(str(g) for g in ms_model.genes if not str(g).startswith("mRNA_"))

for cid in all_cids:
    pub_rxn = pub_rxn_by_cid.get(cid)
    ms_rxn = ms_rxn_by_cid.get(cid)

    membership = "both" if pub_rxn and ms_rxn else ("published" if pub_rxn else "modelseed")

    # Published flux + FVA class
    if pub_rxn:
        pub_flux_val = pub_fluxes.get(pub_rxn.id, 0.0)
        pub_fva_entry = pub_fva.get(pub_rxn.id, {"MIN": 0, "MAX": 0})
        pub_class = _legacy.classify_fva_flux(pub_fva_entry, pub_flux_val)
        pub_flux_str = f"{pub_flux_val:.6g} ({pub_class})"
    else:
        pub_flux_str = ""

    # ModelSEED flux + FVA class
    if ms_rxn:
        ms_flux_val = ms_fluxes.get(ms_rxn.id, 0.0)
        ms_fva_entry = ms_fva.get(ms_rxn.id, {"MIN": 0, "MAX": 0})
        ms_class = _legacy.classify_fva_flux(ms_fva_entry, ms_flux_val)
        ms_flux_str = f"{ms_flux_val:.6g} ({ms_class})"
    else:
        ms_flux_str = ""

    # Genes
    pub_genes = set(str(g) for g in pub_rxn.genes if not str(g).startswith("mRNA_")) if pub_rxn else set()
    ms_genes = set(str(g) for g in ms_rxn.genes if not str(g).startswith("mRNA_")) if ms_rxn else set()
    shared_genes = "; ".join(sorted(pub_genes & ms_genes))

    # Extra published genes
    extra_pub_genes_list = []
    for g in sorted(pub_genes - ms_genes):
        ms_rxns_for_gene = ms_gene_rxn_map.get(g, [])
        if ms_rxns_for_gene:
            extra_pub_genes_list.append(f"{g} ({','.join(ms_rxns_for_gene)})")
        else:
            extra_pub_genes_list.append(g)
    extra_pub_genes = "; ".join(extra_pub_genes_list)

    # Extra modelseed genes
    extra_ms_genes_list = []
    for g in sorted(ms_genes - pub_genes):
        pub_rxns_for_gene = pub_gene_rxn_map.get(g, [])
        if pub_rxns_for_gene:
            extra_ms_genes_list.append(f"{g} ({','.join(pub_rxns_for_gene)})")
        else:
            extra_ms_genes_list.append(g)
    extra_ms_genes = "; ".join(extra_ms_genes_list)

    # Directionality
    if pub_rxn and ms_rxn:
        pub_dir = get_reaction_directionality(pub_rxn)
        ms_dir = get_reaction_directionality(ms_rxn)
        directionality = f"{pub_dir}/{ms_dir}"
    elif pub_rxn:
        directionality = get_reaction_directionality(pub_rxn)
    else:
        directionality = get_reaction_directionality(ms_rxn)

    ref_rxn = pub_rxn if pub_rxn else ms_rxn
    equation = reaction_equation_with_names(ref_rxn)

    # Stoichiometry differences
    if pub_rxn and ms_rxn:
        stoich_result = compare_reaction_stoichiometry(pub_rxn, ms_rxn)
        stoich_parts = []
        for mid in stoich_result["only_in_a"]:
            stoich_parts.append(f"{mid}: missing in modelseed")
        for mid in stoich_result["only_in_b"]:
            stoich_parts.append(f"{mid}: missing in published")
        for mid, (ca, cb) in stoich_result["coefficient_diffs"].items():
            stoich_parts.append(f"{mid}: {ca:g} in published vs {cb:g} in modelseed")
        stoich_diff_str = "; ".join(stoich_parts) if stoich_parts else ""
    else:
        stoich_diff_str = ""

    rows.append({
        "ID": cid,
        "published_pyr_flux": pub_flux_str,
        "modelseed_pyr_flux": ms_flux_str,
        "membership": membership,
        "shared_genes": shared_genes,
        "extra_published_genes": extra_pub_genes,
        "extra_modelseed_genes": extra_ms_genes,
        "directionality": directionality,
        "equation": equation,
        "stoichiometry_differences": stoich_diff_str
    })

df = pd.DataFrame(rows)
print(f"\nComparison dataframe: {len(df)} rows")
print(f"Membership counts:")
print(df["membership"].value_counts().to_string())

session.cache.save("ModelReconciliation/comparison_dataframe", df.to_dict(orient="records"))
df.to_csv(f"nboutput/model_reconciliation.tsv", sep="\t", index=False)
print(f"\nSaved to datacache and nboutput/model_reconciliation.tsv")
display(df)

## Cell 4: Identify Published-Only Reactions with Non-Overlapping Genes

Produces a list of reactions that are:
1. **Only in the published model** (not in ModelSEED model)
2. **Associated with genes that do NOT appear anywhere in the ModelSEED model**
3. **Not exchange reactions** (single-metabolite reactions are excluded)

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()

# Load comparison data
comparison = session.cache.load("ModelReconciliation/comparison_dataframe")

# Reload models for gene set extraction
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
ms_mdlutl = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

pub_model = pub_mdlutl.model
ms_model = ms_mdlutl.model

# Build the set of ALL genes in the ModelSEED model
ms_all_genes = set(str(g) for g in ms_model.genes)

# Identify published-only reactions with genes NOT in ModelSEED model (exclude exchanges)
published_only_unique_gene_rxns = []
for rxn in pub_model.reactions:
    if len(rxn.metabolites) == 1:
        continue
    if rxn.id.startswith("bio") or "GROWTH" in rxn.id or "BIOMASS" in rxn.id:
        continue
    std_id = _legacy.standardize_exchange_id(rxn)
    ms_has_rxn = std_id in [r.id for r in ms_model.reactions] or rxn.id in [r.id for r in ms_model.reactions]
    if ms_has_rxn:
        continue
    rxn_genes = set(str(g) for g in rxn.genes if not str(g).startswith("mRNA_"))
    if not rxn_genes:
        continue
    overlapping_genes = rxn_genes & ms_all_genes
    if len(overlapping_genes) == 0:
        published_only_unique_gene_rxns.append({
            "rxn_id": rxn.id,
            "genes": sorted(rxn_genes),
            "gene_rule": rxn.gene_reaction_rule,
            "equation": reaction_equation_with_names(rxn),
            "lower_bound": rxn.lower_bound,
            "upper_bound": rxn.upper_bound,
        })

print(f"Published-only reactions with genes NOT in ModelSEED model: {len(published_only_unique_gene_rxns)}")
print(f"(Excludes exchange reactions and spontaneous reactions)\n")

df_unique = pd.DataFrame(published_only_unique_gene_rxns)
if len(df_unique) > 0:
    df_unique["genes_str"] = df_unique["genes"].apply(lambda x: "; ".join(x))
    display(df_unique[["rxn_id", "genes_str", "gene_rule", "equation", "lower_bound", "upper_bound"]])

session.cache.save("ModelReconciliation/published_only_unique_gene_rxns", published_only_unique_gene_rxns)
print(f"\nSaved to datacache")

## Cell 5: Save ModelSEED Model Locally as JSON

Pull the ModelSEED model from KBase and save it to `models/ModelSEED_ADP1.json`
so subsequent cells can load it locally without KBase network access.

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
import cobra.io

# Load ModelSEED model from KBase
ms_mdlutl = _legacy.get_model("179225/Abaylyi_ADP1_RASTMS2_OMEGGA_Abaylyi_ADP1_RAST.mdlMS2_OMEGGA_iAbaylyi_Carbon_Succinic.gf")
# Remove mRNA_ genes
gene_remove_list = [g for g in ms_mdlutl.model.genes if str(g.id).startswith("mRNA_")]
for gene in gene_remove_list:
    ms_mdlutl.model.genes.remove(gene)

# Save as COBRApy JSON
output_path = "models/ModelSEED_ADP1.json"
cobra.io.save_json_model(ms_mdlutl.model, output_path)
print(f"ModelSEED model saved to {output_path}")
print(f"  Reactions: {len(ms_mdlutl.model.reactions)}")
print(f"  Metabolites: {len(ms_mdlutl.model.metabolites)}")
print(f"  Genes: {len(ms_mdlutl.model.genes)}")

## Cell 6: Merge Published Reactions and Run ATP Safe Expansion

Takes the published-only reactions with non-overlapping genes (from Cell 4) and adds them
to the ModelSEED model. Then uses `MSATPCorrection` to filter ATP-breaking reactions.

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
from cobra import Reaction, Metabolite
from modelseedpy import MSATPCorrection

# Load ModelSEED model from local JSON (saved in Cell 5)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
print(f"Base ModelSEED model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# Load published model (source of reactions to add)
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

# Load published-only unique gene reactions list from Cell 4
unique_rxn_list = session.cache.load("ModelReconciliation/published_only_unique_gene_rxns")
unique_rxn_ids = set(r["rxn_id"] for r in unique_rxn_list)
print(f"Total candidate reactions from Cell 4: {len(unique_rxn_ids)}")

# Exclude single-compound diffusion reactions
diffusion_rxn_ids = set()
for rxn_id in list(unique_rxn_ids):
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    if is_diffusion_reaction(pub_rxn):
        diffusion_rxn_ids.add(rxn_id)
        unique_rxn_ids.discard(rxn_id)

print(f"Single-compound diffusion reactions excluded: {len(diffusion_rxn_ids)}")
for rid in sorted(diffusion_rxn_ids):
    pub_rxn = pub_model.reactions.get_by_id(rid)
    eq = reaction_equation_with_names(pub_rxn)
    print(f"  {rid}: {eq}")

# Manual exclusions: reactions known to create flux loops
manual_exclusions = {"rxn08856_c0", "rxn00779_c0", "rxn12504_c0", "rxn03630_c0"}
for rxn_id in manual_exclusions:
    if rxn_id in unique_rxn_ids:
        unique_rxn_ids.discard(rxn_id)
        print(f"Manually excluded: {rxn_id} (flux loop)")

print(f"Candidate reactions after filtering: {len(unique_rxn_ids)}")

# --- Step 1: Add published reactions + metabolites + exchange reactions ---
reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in unique_rxn_ids:
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=_legacy.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)
print(f"\nAdded {len(reactions_to_add)} reactions to model")

# Add exchange reactions for any new extracellular metabolites that lack exchanges
for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(
            len(rxn.metabolites) == 1 and met in rxn.metabolites
            for rxn in met.reactions
        )
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)

if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)
    print(f"Added {len(exchange_rxns_to_add)} exchange reactions for new extracellular metabolites")

print(f"Model after additions: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# --- Step 2: Run ATP safe expansion ---
new_rxn_expansion_list = []
for rxn_id in unique_rxn_ids:
    if rxn_id in ms_model.reactions:
        rxn = ms_model.reactions.get_by_id(rxn_id)
        if rxn.upper_bound > 0:
            new_rxn_expansion_list.append([rxn, ">"])
        if rxn.lower_bound < 0:
            new_rxn_expansion_list.append([rxn, "<"])
        rxn.lower_bound = 0
        rxn.upper_bound = 0

print(f"\nReaction directions to test: {len(new_rxn_expansion_list)}")

atp_correction = MSATPCorrection(ms_mdlutl, load_default_medias=True)
print("Evaluating ATP production on default media...")
atp_correction.evaluate_growth_media(no_gapfilling=True)
atp_correction.determine_growth_media()
atp_correction.restore_noncore_reactions()
tests = atp_correction.build_tests()
print(f"Built {len(tests)} ATP tests from {len(atp_correction.selected_media)} selected media")

print("Running ATP safe expansion test on added reactions...")
filtered_rxns = ms_mdlutl.reaction_expansion_test(
    new_rxn_expansion_list, tests, attribute_label="published_rxn_atp_filter"
)

if filtered_rxns is None:
    print("WARNING: No valid solution found")
    filtered_rxn_ids = set()
else:
    filtered_rxn_ids = set()
    for item in filtered_rxns:
        rxn = item[0]
        direction = item[1]
        filtered_rxn_ids.add(rxn.id)
        if direction == ">":
            rxn.upper_bound = 0
        else:
            rxn.lower_bound = 0
        if rxn.lower_bound == 0 and rxn.upper_bound == 0:
            ms_model.remove_reactions([rxn])

    print(f"\nATP expansion filter results:")
    print(f"  Reactions/directions tested: {len(new_rxn_expansion_list)}")
    print(f"  Filtered (ATP-breaking): {len(filtered_rxns)}")
    print(f"  Unique reactions removed or constrained: {len(filtered_rxn_ids)}")

surviving_rxn_ids = unique_rxn_ids - filtered_rxn_ids
still_in_model = set(r["rxn_id"] for r in unique_rxn_list if r["rxn_id"] in [rxn.id for rxn in ms_model.reactions])

print(f"\nFinal merged model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")
print(f"  Published reactions retained: {len(still_in_model)} / {len(unique_rxn_ids)}")

merge_results = {
    "reactions_added": len(unique_rxn_ids),
    "reactions_filtered_atp": len(filtered_rxn_ids) if filtered_rxns else 0,
    "reactions_retained": len(still_in_model),
    "filtered_rxn_ids": sorted(filtered_rxn_ids),
    "retained_rxn_ids": sorted(still_in_model),
    "diffusion_rxn_ids": sorted(diffusion_rxn_ids),
    "manual_exclusion_ids": sorted(manual_exclusions),
    "final_model_reactions": len(ms_model.reactions),
    "final_model_metabolites": len(ms_model.metabolites),
    "final_model_genes": len(ms_model.genes),
}
session.cache.save("ModelReconciliation/merge_results", merge_results)
print("\nMerge results saved to datacache.")

## Cell 7: Run pFBA and FVA on Merged Model in Pyruvate Media

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
from cobra import Reaction, Metabolite

# Rebuild merged model from components (for cell independence)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

merge_results = session.cache.load("ModelReconciliation/merge_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])
print(f"Re-adding {len(retained_rxn_ids)} retained published reactions to ModelSEED model")

# Add retained reactions from published model
reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in retained_rxn_ids:
    if rxn_id in ms_model.reactions:
        continue
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=_legacy.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)

for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(len(rxn.metabolites) == 1 and met in rxn.metabolites for rxn in met.reactions)
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)
if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)

print(f"Merged model: {len(ms_model.reactions)} rxns, {len(ms_model.metabolites)} mets, {len(ms_model.genes)} genes")

# --- Run pFBA ---
print("\n=== Merged Model: pFBA in Pyruvate Media ===")
merged_pfba = _legacy.run_fba(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                               objective="MAX{bio1}", run_pfba=True)
print(f"pFBA growth rate: {merged_pfba.objective_value:.6f}")

# --- Run FVA at 50% optimum ---
merged_fva = _legacy.run_fva(ms_mdlutl, media="KBaseMedia/Carbon-Pyruvic-Acid",
                              objective="MAX{bio1}", fraction_of_optimum=0.5)
print(f"FVA completed for {len(merged_fva)} reactions at 50% optimum")

merged_results = {
    "growth_rate": merged_pfba.objective_value,
    "fluxes": {rxn_id: float(val) for rxn_id, val in merged_pfba.fluxes.items()},
    "fva": merged_fva
}
session.cache.save("ModelReconciliation/merged_pyr_results", merged_results)

pub_results = session.cache.load("ModelReconciliation/published_pyr_results")
ms_results = session.cache.load("ModelReconciliation/modelseed_pyr_results")
print(f"\nGrowth rate comparison:")
print(f"  Published model:  {pub_results['growth_rate']:.6f}")
print(f"  ModelSEED model:  {ms_results['growth_rate']:.6f}")
print(f"  Merged model:     {merged_pfba.objective_value:.6f}")

# Classify merged FVA results
from collections import Counter
fva_classes = {}
for rxn_id in merged_fva:
    flux_val = merged_results["fluxes"].get(rxn_id, 0.0)
    fva_entry = merged_fva[rxn_id]
    fva_classes[rxn_id] = _legacy.classify_fva_flux(fva_entry, flux_val)

class_counts = Counter(fva_classes.values())
print(f"\nMerged model FVA classification:")
for cls, count in sorted(class_counts.items()):
    print(f"  {cls}: {count}")

print("\nMerged model results saved to datacache.")

## Cell 8: Save the Final Merged Model

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
from cobra import Reaction, Metabolite
import cobra.io

# Rebuild the merged model from components (for cell independence)
ms_mdlutl = MSModelUtil.from_cobrapy("models/ModelSEED_ADP1.json")
ms_model = ms_mdlutl.model
pub_mdlutl = MSModelUtil.from_cobrapy("models/FullyTranslatedPublishedModel.json")
pub_model = pub_mdlutl.model

merge_results = session.cache.load("ModelReconciliation/merge_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])

reactions_to_add = []
exchange_rxns_to_add = []
for rxn_id in retained_rxn_ids:
    if rxn_id in ms_model.reactions:
        continue
    pub_rxn = pub_model.reactions.get_by_id(rxn_id)
    new_rxn = Reaction(pub_rxn.id)
    new_rxn.name = pub_rxn.name
    new_rxn.lower_bound = pub_rxn.lower_bound
    new_rxn.upper_bound = pub_rxn.upper_bound
    new_rxn.gene_reaction_rule = pub_rxn.gene_reaction_rule
    met_dict = {}
    for met, coeff in pub_rxn.metabolites.items():
        if met.id in ms_model.metabolites:
            met_dict[ms_model.metabolites.get_by_id(met.id)] = coeff
        else:
            new_met = Metabolite(met.id, name=met.name,
                                 compartment=_legacy.normalize_compartment(met.compartment),
                                 formula=met.formula, charge=met.charge)
            met_dict[new_met] = coeff
    new_rxn.add_metabolites(met_dict)
    reactions_to_add.append(new_rxn)

ms_model.add_reactions(reactions_to_add)

for met in ms_model.metabolites:
    if met.compartment == "e0":
        has_exchange = any(len(rxn.metabolites) == 1 and met in rxn.metabolites for rxn in met.reactions)
        if not has_exchange:
            ex_rxn = Reaction(f"EX_{met.id}")
            ex_rxn.name = f"Exchange for {met.name}"
            ex_rxn.lower_bound = -1000
            ex_rxn.upper_bound = 1000
            ex_rxn.add_metabolites({met: -1})
            exchange_rxns_to_add.append(ex_rxn)
if exchange_rxns_to_add:
    ms_model.add_reactions(exchange_rxns_to_add)

output_path = "models/MergedADP1Model.json"
cobra.io.save_json_model(ms_model, output_path)
print(f"Merged model saved to {output_path}")
print(f"  Reactions: {len(ms_model.reactions)}")
print(f"  Metabolites: {len(ms_model.metabolites)}")
print(f"  Genes: {len(ms_model.genes)}")
print(f"  Published reactions added: {len(retained_rxn_ids)}")

## Cell 9: Escher Map of Merged Model with Reaction Class Badges

Generate an interactive Escher map painted with pyruvate pFBA fluxes, badged by
source (ModelSEED vs Published) and FVA class (Essential, Variable, Blocked).

In [ ]:
%run util.py
from util_legacy import NotebookUtil
_legacy = NotebookUtil()
import os
from collections import Counter

# Load the merged model
merged_mdlutl = MSModelUtil.from_cobrapy("models/MergedADP1Model.json")
merged_model = merged_mdlutl.model

# Load merge results and flux/FVA results
merge_results = session.cache.load("ModelReconciliation/merge_results")
merged_results = session.cache.load("ModelReconciliation/merged_pyr_results")
retained_rxn_ids = set(merge_results["retained_rxn_ids"])

fluxes = merged_results["fluxes"]
fva = merged_results["fva"]

def simplify_fva_class(fva_class):
    if "essential" in fva_class:
        return "Essential"
    elif "blocked" in fva_class:
        return "Blocked"
    else:
        return "Variable"

reaction_classes = {}
for rxn in merged_model.reactions:
    if len(rxn.metabolites) == 1:
        continue
    if rxn.id.startswith("bio") or "GROWTH" in rxn.id:
        continue
    
    source = "Published" if rxn.id in retained_rxn_ids else "ModelSEED"
    
    flux_val = fluxes.get(rxn.id, 0.0)
    fva_entry = fva.get(rxn.id, {"MIN": 0, "MAX": 0})
    detailed_class = _legacy.classify_fva_flux(fva_entry, flux_val)
    simple_class = simplify_fva_class(detailed_class)
    
    reaction_classes[rxn.id] = f"{simple_class} ({source})"

badge_counts = Counter(reaction_classes.values())
print("Reaction badge categories:")
for badge, count in sorted(badge_counts.items()):
    print(f"  {badge}: {count}")
print(f"  Total badged: {len(reaction_classes)}")

output_path = f"nboutput/ModelReconciliation/merged_model_escher.html"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

print("\nSearching for best Escher map...")
map_df = _legacy.list_available_maps(model=merged_mdlutl, as_df=True)
if len(map_df) > 0:
    map_df_sorted = map_df.sort_values("model_reaction_coverage", ascending=False)
    print(f"Top maps by coverage:")
    for _, row in map_df_sorted.head(5).iterrows():
        print(f"  {row['name']} ({row['source']}): {row.get('model_reaction_coverage', 0):.1%} coverage")
    best_map = map_df_sorted.iloc[0]["name"]
else:
    best_map = "core"

print(f"\nUsing map: {best_map}")

_legacy.create_map_html2(
    model=merged_mdlutl,
    map=best_map,
    output_path=output_path,
    flux=fluxes,
    reaction_classes=reaction_classes,
    height=800,
    width=1200
)

print(f"\nEscher map saved to: {output_path}")

session.cache.save("ModelReconciliation/reaction_badges", reaction_classes)